# 9. 텍스트 벡터화, 임베딩과 분류 실습

이번 장에서는 텍스트를 머신러닝 모델이 사용할 수 있는 숫자 표현으로 바꾸는 방법을 다룹니다.  
초기 인코딩 방식에서 시작해 BoW, TF-IDF, N-gram, Word2Vec, 문맥적 임베딩으로 발전해 온 흐름을 살펴보고, 마지막에는 영화리뷰 감성 분류 모델을 직접 학습합니다.


## 1. 텍스트 벡터화의 발전 흐름

텍스트 벡터화는 문자로 된 데이터를 모델이 계산할 수 있는 숫자 표현으로 바꾸는 과정입니다.  
처음에는 단어를 구분하기 위한 단순 인코딩을 사용했고, 이후에는 문서 안의 단어 빈도와 중요도를 반영하는 통계 기반 벡터화가 널리 쓰였습니다.

그 다음 단계에서 Word2Vec 같은 임베딩 기법이 등장했습니다. 임베딩은 단순히 숫자로 바꾸는 것을 넘어, 단어의 의미와 쓰임이 벡터 공간에 반영되도록 학습합니다.

<img src="image/text_vectorization_sparse_dense.svg" width="760">

이미지 출처: 김민수 강사


### 1.1 초기 표현: 정수 인코딩과 원-핫 인코딩

정수 인코딩과 원-핫 인코딩은 임베딩이라기보다, 텍스트를 모델에 넣기 위해 단어를 숫자로 구분하는 초기 표현 방식입니다.

- **정수 인코딩(Integer Encoding)**  
  - 단어마다 번호(ID)만 붙이는 방식입니다.  
    예) `{"강아지": 1, "고양이": 2, "자동차": 3, ...}`  
  - 숫자의 크기와 단어의 의미는 아무 관계가 없는데,  
    모델 입장에서는 `1 < 2 < 3` 같은 **가짜 순서 정보**를 학습할 위험이 있습니다.

- **원-핫 인코딩(One-hot Encoding)**  
  - 단어 사전 크기가 `V`일 때, 길이가 `V`인 벡터에서 해당 단어 위치만 1이고 나머지는 모두 0인 벡터를 사용합니다.
  - 모든 벡터가 서로 직교(orthogonal)라서 **단어 사이의 유사성이 전혀 표현되지 않습니다.**
  - 벡터가 매우 길고 대부분이 0인 **희소(sparse) 벡터**라서, 계산 효율도 좋지 않습니다.

```
어휘 사전: ["강아지", "고양이", "자동차", "사과", "바나나"]
"고양이" = [0, 1, 0, 0, 0]
"강아지" = [1, 0, 0, 0, 0]
→ 두 벡터의 내적 = 0 (완전히 다른 방향)
→ 실제로는 비슷한 동물인데, 벡터 상에서는 아무 관련이 없음
```
정리하면, 초기 인코딩은 “컴퓨터가 처리할 수 있도록 텍스트를 숫자로 바꾸는 것”에는 성공했지만,  
“단어의 의미나 단어들 사이의 관계를 표현하는 것”에는 한계가 있습니다.


### 1.2 한계를 보완해 온 방향

텍스트 표현 방법은 앞선 방식의 한계를 보완하는 방향으로 발전했습니다.

| 단계 | 대표 방법 | 해결하려는 문제 | 남는 한계 |
|------|-----------|----------------|----------|
| 단어 구분 | 정수 인코딩, 원-핫 인코딩 | 텍스트를 숫자로 바꿈 | 의미와 유사성 반영 불가 |
| 단어 빈도 반영 | BoW | 문서에 어떤 단어가 많이 나왔는지 표현 | 순서와 문맥 무시 |
| 단어 중요도 반영 | TF-IDF | 너무 흔한 단어의 영향 줄이기 | 깊은 의미와 유사성은 부족 |
| 짧은 순서 반영 | N-gram | 연속된 단어 패턴 일부 반영 | 차원 수 증가 |
| 의미 학습 | Word2Vec 등 임베딩 | 비슷한 문맥의 단어를 가까운 벡터로 학습 | 단어 의미가 문맥마다 바뀌는 문제 |
| 문맥 반영 | BERT, GPT 계열 문맥적 임베딩 | 문장 안에서 단어 의미를 동적으로 계산 | 모델 구조와 계산량이 커짐 |

이 장도 이 순서에 맞춰, 단순한 표현에서 의미를 학습하는 표현으로 이동합니다.

### 1.3 인코딩, 벡터화, 임베딩 구분

| 구분 | 대표 방법 | 핵심 아이디어 |
|------|-----------|---------------|
| 인코딩 | 정수 인코딩, 원-핫 인코딩 | 단어를 서로 구분할 수 있게 숫자나 희소 벡터로 바꿈 |
| 통계 기반 벡터화 | BoW, TF-IDF, N-gram | 문서 안의 등장 횟수, 중요도, 짧은 순서를 숫자로 표현 |
| 임베딩 | Word2Vec, BERT/GPT 계열 문맥적 임베딩 | 데이터로부터 의미와 문맥이 반영된 밀집 벡터를 학습 |

따라서 정수 인코딩과 원-핫 인코딩은 임베딩의 한 종류라기보다, 임베딩 이전 단계의 단순한 숫자화 방법으로 보는 편이 정확합니다.

이 차시에서는 텍스트 표현 방법을 발전 흐름에 따라 정리합니다.

- **통계 기반 벡터화(BoW / TF-IDF / N-gram)**  
  단어 빈도와 중요도를 이용해 문서를 숫자 벡터로 표현합니다.

- **분산 표현(Word2Vec)**  
  비슷한 문맥에서 쓰이는 단어가 비슷한 벡터를 갖도록 학습합니다.

- **텍스트 분류 베이스라인**  
  벡터화한 문장을 머신러닝 모델에 넣어 감성 분류 흐름을 완성합니다.


## 2. 통계 기반 벡터화: BoW / TF-IDF / N-gram 

초기 인코딩은 단어를 구분할 수는 있지만, 문서 안에서 어떤 단어가 얼마나 중요하게 쓰였는지는 표현하지 못합니다.  
이 한계를 보완하기 위해 **통계(statistics)에 기반한 벡터화 기법**들이 널리 사용되었습니다.

- 이 기법들은 **“임베딩”이라고 부르기보다는**,  
  “단어/문서의 등장 패턴을 숫자로 표현한 벡터화 방법”에 가깝습니다.
- 즉, **의미를 잘 담은 밀집 임베딩이라기보다는**,  
  “등장 횟수·빈도·조합”을 수치화한 **전통적인 표현 방식**이라고 보는 것이 더 정확합니다.

이 섹션에서는 대표적인 통계 기반 벡터화 기법인 **Bag of Words(BoW), TF-IDF, N-gram**을 순서대로 살펴보고,  
이후 등장하는 **신경망 기반 임베딩**과 어떤 차이가 있는지 연결해 보겠습니다.

<img src="image/bow_tfidf_ngram_flow.svg" width="760">

이미지 출처: 김민수 강사


#### 1) Bag of Words (BoW)

**아이디어**  
- 문서 안에 어떤 단어들이 **몇 번** 등장했는지만 세어서 벡터로 만드는 방식
- 단어의 **순서나 문맥은 완전히 무시**하고, “가방 속에 단어 개수만 넣어둔다”고 생각하면 됩니다.


```text
문장1: "고양이가 물을 마신다"
문장2: "물이 고양이를 마신다"

어휘 사전: ["고양이", "물", "마신다"]

→ 문장1 BoW: [1, 1, 1]
→ 문장2 BoW: [1, 1, 1]

두 문장의 단어 순서와 의미는 분명히 다르지만,
BoW 벡터는 완전히 동일하게 나옵니다.
```

- 장점:  
  - 구현이 단순하고 직관적
  - 전통적인 문서 분류, 스팸 필터링 등에서 오랫동안 잘 동작해 옴
- 한계:  
  - 단어 **순서, 문맥, 구문 구조**를 전혀 반영하지 못함  
  - 단어 사전 크기만큼 차원이 필요 → **고차원 희소 벡터(sparse vector)**

#### 2) TF-IDF (Term Frequency – Inverse Document Frequency)
TF-IDF는 문서를 **단어별 중요도 점수표**로 바꾸는 방법입니다.  
단어가 한 문서 안에서 자주 나오면서도, 다른 문서에는 흔하지 않다면 그 문서에서 중요한 단어로 봅니다.

실무에서는 다음처럼 자주 사용합니다.
- 검색: 사용자의 검색어와 문서가 얼마나 관련 있는지 계산
- 문서 분류: 뉴스, 리뷰, 문의글을 분류 모델에 넣기 위한 입력값 생성
- 추천/유사도: 문서 벡터끼리 비교해 비슷한 문서 찾기
- 키워드 추출: 문서별로 특징적인 단어 확인

형태는 보통 **행은 문서, 열은 단어, 값은 TF-IDF 점수**인 표입니다.

<table style="font-size:16px; border-collapse:collapse; min-width:760px;">
  <thead>
    <tr>
      <th style="border:1px solid #bbb; padding:8px;">문서</th>
      <th style="border:1px solid #bbb; padding:8px;">배송</th>
      <th style="border:1px solid #bbb; padding:8px;">환불</th>
      <th style="border:1px solid #bbb; padding:8px;">품질</th>
      <th style="border:1px solid #bbb; padding:8px;">상담</th>
      <th style="border:1px solid #bbb; padding:8px;">지연</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border:1px solid #bbb; padding:8px;">리뷰1: 배송이 빨랐고 품질도 좋다</td>
      <td style="border:1px solid #bbb; padding:8px; font-weight:bold;">0.52</td>
      <td style="border:1px solid #bbb; padding:8px;">0.00</td>
      <td style="border:1px solid #bbb; padding:8px; font-weight:bold;">0.61</td>
      <td style="border:1px solid #bbb; padding:8px;">0.00</td>
      <td style="border:1px solid #bbb; padding:8px;">0.00</td>
    </tr>
    <tr>
      <td style="border:1px solid #bbb; padding:8px;">리뷰2: 환불 상담이 너무 느리다</td>
      <td style="border:1px solid #bbb; padding:8px;">0.00</td>
      <td style="border:1px solid #bbb; padding:8px; font-weight:bold;">0.58</td>
      <td style="border:1px solid #bbb; padding:8px;">0.00</td>
      <td style="border:1px solid #bbb; padding:8px; font-weight:bold;">0.58</td>
      <td style="border:1px solid #bbb; padding:8px;">0.00</td>
    </tr>
    <tr>
      <td style="border:1px solid #bbb; padding:8px;">리뷰3: 배송 지연 때문에 불만이다</td>
      <td style="border:1px solid #bbb; padding:8px; font-weight:bold;">0.45</td>
      <td style="border:1px solid #bbb; padding:8px;">0.00</td>
      <td style="border:1px solid #bbb; padding:8px;">0.00</td>
      <td style="border:1px solid #bbb; padding:8px;">0.00</td>
      <td style="border:1px solid #bbb; padding:8px; font-weight:bold;">0.72</td>
    </tr>
  </tbody>
</table>

위 표처럼 각 리뷰는 숫자 벡터가 됩니다.  
`리뷰1`은 `배송`, `품질` 쪽 점수가 높고, `리뷰2`는 `환불`, `상담` 쪽 점수가 높습니다. 이런 숫자 벡터를 분류 모델에 넣으면 긍정/부정, 주제, 문의 유형 등을 예측할 수 있습니다.

- BoW는 단순히 "단어가 몇 번 나왔는가"만 봅니다.  
- TF-IDF는 "그 문서에서 자주 나왔는가"와 "다른 문서에는 덜 흔한가"를 함께 봅니다.

예를 들어 거의 모든 리뷰에 등장하는 `영화`, `정말` 같은 단어는 점수가 낮아지고, 특정 문서에만 두드러지는 `환불`, `지연`, `품질` 같은 단어는 점수가 높아질 수 있습니다.

#### 구성 요소

1. **TF (Term Frequency)**  
   - 특정 단어 *t*가 문서 *d* 안에서 얼마나 자주 등장했는지를 나타냄  
   - 단순히 "등장 횟수"를 쓰기도 하고, 문서 길이를 고려해 "등장 비율"로 정규화하기도 함  
   - 자주 나온 단어가 높은 점수  

   **공식**  
   $$
   TF(t, d) = \frac{\text{단어 t가 문서 d에 등장한 횟수}}{\text{문서 d의 전체 단어 수}}
   $$


2. **IDF (Inverse Document Frequency)**  
   - 특정 단어가 전체 문서 집합에서 얼마나 희귀한지를 나타냄  
   - 흔하게 나오는 단어는 IDF가 0에 가까워지고, 드물게 나오는 단어는 IDF가 커짐

   **공식**  
   $$
   IDF(t) = \log \frac{N(전체문서수)}{df(t)}
   $$

   - $ N $: 전체 문서 수  
   - $ df(t) $: 단어 t가 등장한 문서의 수(document frequency)  
   - 즉, 다른 문서에 거의 등장하지 않는 단어일수록 높은 점수



3. **TF-IDF**  
   - 최종적으로 TF와 IDF를 곱해 단어의 중요도를 산출  
   - 다른 문서에 자주 등장하지 않는 단어가 문서A에서 자주 등장했다면, 해당 단어는 문서 A에서 높은 TF-IDF 스코어 획득 

   **공식**  
   $$
   TF\text{-}IDF(t, d) = TF(t, d) \times IDF(t)
   $$

#### 계산 예시

문서 집합:
- 문서1: `"나는 고양이를 좋아한다"` (총 3단어)
- 문서2: `"나는 개를 좋아한다"` (총 3단어)  
전체 문서 수 $N=2$  
어휘: {"고양이", "개", "좋아한다", "나는"}

#### Step 1. IDF 계산 (단어의 공통 중요도)

| 단어 | 문서 빈도 (df, Document Frequency) | IDF 계산 (\log(N/df)) | 결과 (\log_{10} 사용) |
| :--- | :--- | :--- | :--- |
| **"고양이"** | 문서1에만 등장 $\rightarrow$ df=1 | $\log (2/1)$ | $\approx 0.301$ |
| **"개"** | 문서2에만 등장 $\rightarrow$ df=1 | $\log (2/1)$ | $\approx 0.301$ |
| **"좋아한다"** | 문서1, 문서2 모두 등장 $\rightarrow$ df=2 | $\log (2/2) = \log 1$ | $0$ |
| **"나는"** | 문서1, 문서2 모두 등장 $\rightarrow$ df=2 | $\log (2/2) = \log 1$ | $0$ |

#### Step 2. 문서1의 TF-IDF 벡터 계산 (문서1: "나는 고양이를 좋아한다")

| 단어 | TF (Term Frequency, 1/3) | IDF (Step 1 결과) | TF-IDF (TF $\times$ IDF) |
| :--- | :--- | :--- | :--- |
| **"고양이"** | $1/3 \approx 0.33$ | $0.301$ | $\approx 0.0993$ |
| **"개"** | $0$ | $0.301$ | $0$ |
| **"좋아한다"** | $1/3 \approx 0.33$ | $0$ | $0$ |
| **"나는"** | $1/3 \approx 0.33$ | $0$ | $0$ |
<br>

**결과: 문서1의 TF-IDF 벡터** $\approx [0.0993, 0, 0, 0]$ (어휘 순)

#### Step 3. 문서2의 TF-IDF 벡터 계산 (문서2: "나는 개를 좋아한다")

| 단어 | TF (Term Frequency, 1/3) | IDF (Step 1 결과) | TF-IDF (TF $\times$ IDF) |
| :--- | :--- | :--- | :--- |
| **"고양이"** | $0$ | $0.301$ | $0$ |
| **"개"** | $1/3 \approx 0.33$ | $0.301$ | $\approx 0.0993$ |
| **"좋아한다"** | $1/3 \approx 0.33$ | $0$ | $0$ |
| **"나는"** | $1/3 \approx 0.33$ | $0$ | $0$ |

**결과: 문서2의 TF-IDF 벡터** $\approx [0, 0.0993, 0, 0]$ (어휘 순)

**최종 결론:**
TF-IDF는 이 과정을 통해 다음을 달성합니다.
1.  **공통 단어("나는", "좋아한다")**: 두 문서에 흔하게 나타나므로 $\text{IDF}$가 $0$이 되어 **중요도 0**을 부여합니다.
2.  **핵심 단어("고양이", "개")**: 해당 문서에서만 특별하게 나타나므로 $\text{IDF}$가 높아져 **높은 중요도**를 부여합니다.

결과적으로, TF-IDF는  
문서1의 특징을 **고양이**로,  
문서2의 특징을 **개**로  
**명확하게 벡터 안에 표현**해 줍니다.

TF-IDF는 통계 기반 벡터화 방식 중 가장 널리 쓰였으며, 특히 다음과 같은 분야에서 핵심적인 역할을 수행했습니다.

1. **정보 검색 (Information Retrieval):**
- 사용자가 검색한 **쿼리(Query)의 중요도**를 계산하고, 웹 문서나 데이터베이스의 **문서 중요도**를 계산하는 데 사용되었습니다. 
- TF-IDF 값이 높은 문서일수록 사용자의 검색 의도와 가장 관련이 높다고 판단하는 방식입니다.
2. **문서 분류 및 클러스터링 (Clustering):**
- 뉴스 기사나 메일 같은 문서들을 TF-IDF 벡터로 변환한 후, 이 벡터들 간의 유사도(거리)를 계산하여   
비슷한 주제의 문서끼리 묶거나(클러스터링),  문서를 특정 주제로 분류하는(Classification)  
**특징 벡터(Feature Vector)** 로 사용되었습니다.

TF-IDF는 단순하지만 강력한 성능을 보여주며, 이후 **단어 중요도 측정** 방식의 기본 토대가 되었습니다.

#### 요약
- TF: 단어가 문서 내에서 얼마나 자주 등장했는가  
- IDF: 단어가 전체 문서 집합에서 얼마나 희귀한가  
- TF-IDF: "문서 내 중요도 × 전체 문서 희소성"  
- 장점: 문서 핵심 키워드 뽑기에 유용, 검색 엔진·문서 추천· 분류에 널리 활용  
- 한계: 여전히 단어 순서·문맥은 반영하지 못함  

#### TF-IDF 실습

아래 실습에서는 `TfidfVectorizer`가 문서 목록을 읽고, 단어 사전을 만든 뒤, 각 문서를 TF-IDF 점수 벡터로 변환하는 과정을 확인합니다.  
옵션을 바꾸면 어떤 단어를 남길지, 몇 개 단어까지 사용할지, N-gram을 함께 사용할지도 조절할 수 있습니다.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer  # 문서를 TF-IDF 벡터로 바꾸는 도구입니다.
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.

corpus = [
    "I love natural language processing",
    "I love programming in Python",
    "natural language processing is exciting",
    "Harry Potter is a series of seven fantasy fantasy novels",
]  # 벡터화할 문서 목록입니다. 여기서는 문장 1개를 문서 1개로 봅니다.

tfidf = TfidfVectorizer(
    stop_words="english",  # 영어 불용어를 제거합니다. 예: "the", "is", "a", "of"
    lowercase=True,  # 대문자를 소문자로 통일합니다. 기본값도 True입니다.
    max_features=None,  # 사용할 단어 수의 상한입니다. None이면 제한하지 않습니다.
    min_df=1,  # 최소 1개 문서에 등장한 단어를 사용합니다. 값을 키우면 너무 드문 단어를 제거합니다.
    max_df=1.0,  # 전체 문서의 100%까지 등장한 단어를 허용합니다. 0.9처럼 낮추면 너무 흔한 단어를 제거합니다.
    ngram_range=(1, 1),  # (1, 1)은 단어 1개씩만 사용합니다. (1, 2)는 단어 1개와 2개 묶음을 함께 사용합니다.
)  # TF-IDF 벡터화 객체를 만듭니다. 아직 실제 학습은 하지 않은 상태입니다.

X_tfidf = tfidf.fit_transform(corpus)  # fit은 단어 사전을 학습하고, transform은 문서를 숫자 벡터로 바꿉니다.

print("단어 사전:", tfidf.vocabulary_)  # 단어가 어느 열 번호에 배정되었는지 확인합니다.
pd.DataFrame(
    X_tfidf.toarray(),  # 희소 행렬을 보기 쉬운 일반 배열로 바꿉니다.
    columns=tfidf.get_feature_names_out(),  # 각 열 이름을 단어로 표시합니다.
)  # 문서-단어 TF-IDF 점수표를 확인합니다.

#### 3) N-gram

BoW/TF-IDF는 단어의 **순서**를 완전히 무시한다는 큰 한계가 있습니다.  
이를 조금이라도 보완하기 위해 등장한 것이 **N-gram**입니다.

- 연속된 N개의 단어(또는 문자)를 하나의 묶음으로 보는 방법
- `CountVectorizer`뿐 아니라 `TfidfVectorizer`에도 `ngram_range` 옵션으로 함께 적용할 수 있습니다.
- 예: 2-gram(바이그램), 3-gram(트라이그램)

```text
문장: "자연어 처리를 열심히 공부한다"

2-gram (바이그램):
["자연어 처리를", "처리를 열심히", "열심히 공부한다"]
```

- 장점:
  - 연속된 단어 패턴을 반영 → **문맥의 일부**를 표현 가능
  - 예: “뉴욕”, “인공 지능”, “자연어 처리” 같은 구를 잘 잡아낼 수 있음
- 한계:
  - N을 크게 할수록 경우의 수가 폭발 → 차원 수가 매우 커짐
  - 희귀하게 등장하는 N-gram은 데이터가 거의 없어 다루기 어려움
  - 여전히 **의미/유사도**를 직접적으로 학습한다기보다는, “공동 등장 패턴”에 가깝다

#### TF-IDF에 N-gram 함께 적용하기

N-gram은 TF-IDF와 함께 자주 사용됩니다. 예를 들어 감성 분석에서는 `not good`, `very bad`, `natural language`처럼 두 단어가 붙어 있을 때 의미가 더 분명해지는 경우가 많습니다.

`ngram_range=(1, 2)`는 단어 1개짜리 표현과 단어 2개짜리 표현을 모두 TF-IDF 열로 만든다는 뜻입니다.


In [ ]:
tfidf_ngram = TfidfVectorizer(
    stop_words="english",  # 영어 불용어를 제거합니다.
    ngram_range=(1, 2),  # unigram(단어 1개)과 bigram(연속 단어 2개)을 함께 사용합니다.
    max_features=20,  # 예시를 보기 쉽게 최대 20개 표현만 남깁니다.
)  # TF-IDF와 N-gram을 동시에 적용하는 벡터화 객체입니다.

X_tfidf_ngram = tfidf_ngram.fit_transform(corpus)  # 단어/구 표현 사전을 학습하고 벡터로 변환합니다.

pd.DataFrame(
    X_tfidf_ngram.toarray(),
    columns=tfidf_ngram.get_feature_names_out(),
)  # 단어 1개 표현과 단어 2개 표현이 함께 열로 만들어졌는지 확인합니다.


#### 4) 통계 기반 벡터화 정리

지금까지 본 BoW / TF-IDF / N-gram은

- 텍스트를 벡터로 만드는 **전통적인 통계 기반 표현 방식**이고,
- “벡터화”는 했지만, 이후에 살펴볼  
  **“의미를 잘 반영하는 밀집 임베딩”과는 거리가 있습니다.**

정리하자면,

- 장점:
  - 단순하고 구현이 쉽다
  - 문서 분류, 키워드 추출, 검색 등 고전적인 NLP 태스크에서 여전히 유용하다
- 근본적인 한계:
  - 단어 순서, 문맥, 깊은 의미를 잘 반영하지 못한다
  - 고차원 희소 벡터가 되어 계산·메모리 효율이 떨어진다
  - 비슷한 의미의 단어를 **벡터 공간에서 가깝게 모으는 구조가 아니다**

이런 이유로,  
> “단어/문장의 **의미**를 벡터 공간에 잘 담아내고 싶다”는 요구가 커지면서  
> **신경망 기반의 임베딩(Word2Vec 등)** 이 본격적으로 사용되기 시작했습니다.

이제 단어의 의미를 벡터 공간에 학습하는 대표적인 방법인  
**Word2Vec**의 기본 아이디어를 살펴보겠습니다.


#### 문제 1. TF-IDF
문장 리스트 `["나는 NLP를 좋아한다", "NLP는 재미있다"]`를 TF-IDF 벡터로 변환하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.feature_extraction.text import TfidfVectorizer  # 문장을 TF-IDF 벡터로 바꾸는 도구입니다.

docs = ["나는 NLP를 좋아한다", "NLP는 재미있다"]  # 벡터화할 문장 목록입니다.
vectorizer = TfidfVectorizer()  # 문장을 숫자 벡터로 바꾸는 객체입니다.
X = vectorizer.fit_transform(docs)  # 학습과 변환을 한 번에 수행합니다.

print(vectorizer.get_feature_names_out())  # 결과를 화면에 출력합니다.
print(X.toarray())  # 결과를 화면에 출력합니다.

```
</details>

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF 벡터화 도구입니다.

docs = ["나는 NLP를 좋아한다", "NLP는 재미있다"]

#### 문제 2. BoW 벡터 만들기
문장 리스트 `["나는 밥을 먹었다", "나는 물을 마셨다"]`에서 **Bag of Words** 벡터를 만들어보세요.  

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.feature_extraction.text import CountVectorizer  # 문장을 단어 개수 벡터로 바꾸는 도구입니다.

docs = ["나는 밥을 먹었다", "나는 물을 마셨다"]  # 벡터화할 문장 목록입니다.
vectorizer = CountVectorizer()  # 문장을 숫자 벡터로 바꾸는 객체입니다.
X = vectorizer.fit_transform(docs)  # 학습과 변환을 한 번에 수행합니다.

print("단어 사전:", vectorizer.get_feature_names_out())  # 결과를 화면에 출력합니다.
print("BoW 벡터:\n", X.toarray())  # 결과를 화면에 출력합니다.

```
</details>

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer  # 단어 빈도 벡터화 도구입니다.

docs = ["나는 밥을 먹었다", "나는 물을 마셨다"]

#### 문제 3. TF-IDF 계산
문장 리스트 `["나는 NLP를 공부한다", "NLP는 재미있다", "공부는 힘들다"]`를 **TF-IDF**로 벡터화하세요.  

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.feature_extraction.text import TfidfVectorizer  # 문장을 TF-IDF 벡터로 바꾸는 도구입니다.

docs = ["나는 NLP를 공부한다", "NLP는 재미있다", "공부는 힘들다"]  # 벡터화할 문장 목록입니다.
vectorizer = TfidfVectorizer()  # 문장을 숫자 벡터로 바꾸는 객체입니다.
X = vectorizer.fit_transform(docs)  # 학습과 변환을 한 번에 수행합니다.

print("단어 사전:", vectorizer.get_feature_names_out())  # 결과를 화면에 출력합니다.
print("TF-IDF 벡터:\n", X.toarray())  # 결과를 화면에 출력합니다.

```
</details>

In [ ]:
# 여기에 작성하세요
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF 벡터화 도구입니다.

docs = ["나는 NLP를 공부한다", "NLP는 재미있다", "공부는 힘들다"]

#### 문제 4. N-gram 적용
문장 `"자연어 처리를 공부한다"`에 대해 **2-gram(바이그램)** 을 추출해보세요.  

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.feature_extraction.text import CountVectorizer  # 문장을 단어 개수 벡터로 바꾸는 도구입니다.

docs = ["자연어 처리를 공부한다"]  # 벡터화할 문장 목록입니다.
vectorizer = CountVectorizer(ngram_range=(2,2))  # 문장을 숫자 벡터로 바꾸는 객체입니다.
X = vectorizer.fit_transform(docs)  # 학습과 변환을 한 번에 수행합니다.

print("바이그램 사전:", vectorizer.get_feature_names_out())  # 결과를 화면에 출력합니다.
print("바이그램 벡터:", X.toarray())  # 결과를 화면에 출력합니다.

```
</details>


In [ ]:
# 여기에 작성하세요
from sklearn.feature_extraction.text import CountVectorizer  # 단어 빈도 벡터화 도구입니다.

docs = ["자연어 처리를 공부한다"]

## 3. 신경망 기반 임베딩: Word2Vec

앞에서 본 BoW, TF-IDF, N-gram은 단어의 등장 패턴을 세거나 가중치로 표현하는 방식이었습니다.  
이번에는 단어의 의미가 벡터 공간에 배치되도록 학습하는 대표 기법인 **Word2Vec**을 살펴봅니다.


#### 1) 분산 표현(Distributed Representation)이란?

기존 통계 기반 벡터화(BoW, TF-IDF)는

- 단어마다 “등장 횟수/빈도/조합”을 세어 벡터로 만들었습니다.
- 단어 의미나 문맥을 직접 학습한다기보다, **통계를 정리한 결과**에 가깝습니다.

반면, 분산 표현은

> “비슷한 문맥에서 쓰이는 단어는 비슷한 벡터를 갖도록”  
> 신경망으로 **벡터 자체를 학습**하는 접근입니다.  
> (의미가 여러 차원에 분산되어 있다는 뜻. 원핫은 오직 한 차원('1'이 있는 그 위치)에만 정보가 있음)

- 각 단어는 보통 수십~수백 차원의 **밀집 벡터(dense vector)** 로 표현되고,
- 벡터 공간에서의 위치/거리/각도가 의미 정보를 반영하도록 학습됩니다.

#### 2) Word2Vec

Word2Vec은 **아주 얕은 신경망**을 사용해 단어 임베딩을 학습하는 모델입니다.

핵심 아이디어:

- 단어는 **주변 단어(문맥)** 과 함께 나타납니다.
- “비슷한 문맥에서 등장하는 단어들끼리는 의미가 비슷할 가능성이 크다.”
- 따라서, 문맥 예시들을 많이 보여주면서  
  임베딩이 그 패턴을 잘 예측하도록 학습합니다.
- 이 예측 문제를 반복해서 풀다 보면, 모델은 단어를 아무 숫자가 아니라 **의미와 쓰임이 반영된 임베딩 벡터**로 배치하는 능력을 갖게 됩니다.

주요 구조는 두 가지입니다. 먼저 큰 방향을 보면, CBOW와 Skip-Gram은 **예측 방향**이 서로 반대입니다.

<img src="image/word2vec_cbow_skipgram_flow.svg" width="760">

이미지 출처: 김민수 강사

아래 그림은 같은 내용을 신경망 구조에 조금 더 가깝게 보여줍니다.

<table>
  <tr>
    <td align="center"><img src="image/word2vec_cbow_commons.svg" width="360"></td>
    <td align="center"><img src="image/word2vec_skipgram_commons.svg" width="360"></td>
  </tr>
  <tr>
    <td align="center"><b>CBOW</b>: 주변 단어로 중심 단어 예측</td>
    <td align="center"><b>Skip-Gram</b>: 중심 단어로 주변 단어 예측</td>
  </tr>
</table>

외부 이미지 출처: Wikimedia Commons, [Continuous Bag of Words model (CBOW).svg](https://commons.wikimedia.org/wiki/File:Continuous_Bag_of_Words_model_(CBOW).svg), [Skip-gram.svg](https://commons.wikimedia.org/wiki/File:Skip-gram.svg) / 저자: Zhang, Aston; Lipton, Zachary C.; Li, Mu; Smola, Alexander J. / 라이선스: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) / 원본 SVG를 파일명만 바꿔 사용

- **CBOW (Continuous Bag of Words)**  
  - 주변 단어들(컨텍스트) → 중심 단어를 예측  
  - 예: `"나는 ___을 먹었다"` → 빈칸에 들어갈 단어를 맞히도록 학습
- **Skip-Gram**  
  - 중심 단어 → 주변 단어들을 예측  
  - 예: `"밥"` → (나는, 먹었다) 같은 주변 단어들을 맞히도록 학습

즉 Word2Vec의 직접 목표는 주변 단어를 맞히는 것이지만, 그 과정에서 얻는 결과물은 **단어 의미를 담은 벡터 표현**입니다.  
학습이 잘 되면 임베딩 벡터 공간에서 재미있는 **벡터 연산**도 가능해지는 것으로 유명합니다.

```text
벡터("왕") - 벡터("남자") + 벡터("여자") ≈ 벡터("여왕")
```

<img src="image/word_vector_illustration_commons.jpg" width="460">

외부 이미지 출처: Wikimedia Commons, [Word vector illustration.jpg](https://commons.wikimedia.org/wiki/File:Word_vector_illustration.jpg) / 저자: Singerep / 라이선스: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) / 원본 이미지를 파일명만 바꿔 사용

즉, 단어들 사이의 관계(성별, 직업, 국가 등)가  
벡터 공간에서 **산술 연산**으로도 어느 정도 드러납니다. 


#### 3) Word2Vec 학습 후 남는 것

Word2Vec은 주변 단어를 맞히는 예측 문제로 학습하지만, 수업에서 가져가야 할 핵심 결과물은 **단어별 임베딩 벡터표**입니다.

- 학습이 끝나면 각 단어마다 고정 길이 숫자 벡터가 만들어집니다.
- `model.wv["학교"]`처럼 특정 단어의 벡터를 꺼내 볼 수 있습니다.
- 비슷한 문맥에서 자주 등장한 단어들은 벡터 공간에서도 가까운 위치에 놓입니다.
- 이렇게 얻은 벡터는 유사 단어 찾기, 시각화, 다른 머신러닝 모델의 입력값 등에 활용할 수 있습니다.

즉 Word2Vec은 단어를 단순한 번호가 아니라, **의미와 쓰임이 반영된 숫자 벡터**로 바꾸는 방법입니다.


## 4. 현대 NLP에서의 임베딩: "고정된 벡터"에서 "계산되는 표현"으로

우리가 지금까지 배운 Word2Vec은 단어마다 하나의 고유한 벡터 좌표를 할당하는 **정적(Static) 임베딩**이었습니다.  
하지만 현대 NLP는 여기서 한 단계 더 진보하여, 단어의 벡터를 문장 속에서 실시간으로 만들어내는 **문맥적 임베딩(Contextual Embedding)** 시대로 넘어왔습니다.

<img src="image/static_vs_contextual_embedding.svg" width="760">

이미지 출처: 김민수 강사


#### 1) 학습 메커니즘의 진화: 빈칸 채우기(Masking)를 통한 문맥 파악

현대 모델은 단순히 단어의 짝을 맞히는 것을 넘어, 문장 전체의 구조를 이해하기 위해 훨씬 고차원적인 방식으로 사전 학습(Pre-training)됩니다.

- **마스크 학습 (Masked Language Modeling):**
  - 문장 중간의 단어를 가려놓고(`[MASK]`), 주변 단어들을 조합해 그 빈칸에 들어갈 단어를 맞히는 연습을 합니다.
  - 예: `"나는 오늘 [MASK]에 가서 돈을 입금했다"` → 주변의 '돈', '입금'을 보고 `[MASK]`가 '은행'임을 추론해야 합니다.
- **문맥 정보의 내재화:** 
  - 이 빈칸을 맞히기 위해 모델은 **"주변에 어떤 단어가 올 때 이 단어의 의미가 어떻게 변하는지"** 그 관계(Attention)를 아주 정교하게 학습하게 됩니다.
  - 이 과정이 수억 개의 문장에 대해 반복되면서, 단어 벡터는 단순한 의미를 넘어 **문맥을 읽어내는 능력**을 갖추게 됩니다.

#### 2) 정적 임베딩 vs 문맥적 임베딩 비교 요약

| 구분 | 정적 임베딩 (Word2Vec) | 문맥적 임베딩 (BERT, GPT 등) |
| :--- | :--- | :--- |
| **핵심 원리** | 미리 저장된 벡터를 **꺼내오기(Lookup)** | 주변 단어와 정보를 **섞어서 계산(Compute)** |
| **다의어 처리** | "은행"의 모든 의미가 하나의 벡터에 섞임 | 문맥(주변 단어)을 보고 실시간으로 의미 분리 |
| **학습 방식** | 단어 간의 단순 통계적 유사성 학습 | **빈칸 채우기(Masking)** 등으로 문맥 간 관계 학습 |
| **구성 요소** | 단어의 고유 의미 | **토큰 의미 + 순서(Position) + 문맥 정보** |

**정리:** 현대 NLP 임베딩의 핵심은 **학습 시점에 빈칸 채우기 등을 통해 단어들 사이의 고차원적인 관계를 미리 파악하고,  
이를 바탕으로 실제 문장에서 단어의 의미를 실시간으로 계산해낸다**는 점에 있습니다.

이러한 '실시간 계산 엔진'의 정체인 **Transformer**와 **Attention**의 구조를 다음 장에서 본격적으로 파헤쳐 보겠습니다.


#### 정리
- **원-핫 인코딩**: 단어 구분만 가능, 유사성 반영 불가  
- **통계 기반 벡터화**: 빈도·중요도·순서를 반영하지만, 의미는 여전히 부족  
- **분산 표현(Word2Vec)**: 신경망을 통해 **단어 간 의미 관계**까지 수치화  
- → 이는 이후 문맥적 임베딩과 LLM을 이해하는 기초 표현으로 연결됨 


#### 문제 5. Word2Vec 학습
문장 리스트 `["나는 학교에 간다", "학교에서 공부한다", "나는 밥을 먹었다"]`로 Word2Vec 모델을 학습한 뒤, `"학교"`와 가장 유사한 단어를 찾아보세요.  

사용할 문법 예시는 다음과 같습니다.

```python
from gensim.models import Word2Vec

# Word2Vec에는 문장을 단어 리스트로 나누어 넣습니다.
sentences = [["나는", "학교에", "간다"], ["학교에서", "공부한다"]]

# vector_size: 단어 벡터 차원, window: 주변 몇 단어까지 볼지, min_count: 최소 등장 횟수
# sg=0은 CBOW, sg=1은 Skip-Gram 방식입니다.
model = Word2Vec(sentences, vector_size=50, window=3, min_count=1, sg=0)

# 학습된 모델에서 특정 단어와 가까운 단어를 찾습니다.
model.wv.most_similar("학교", topn=1)
```

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from gensim.models import Word2Vec  # Word2Vec 임베딩 모델입니다.

sentences = [  # 임베딩 학습에 사용할 문장 목록입니다.
    ["나는", "학교에", "간다"],
    ["학교에서", "공부한다"],
    ["나는", "밥을", "먹었다"],
    ["좋은", "학교", "수업"],
    ["맛있는", "밥"],
    ["시원한", "콜라를","마셨다"],
]

model = Word2Vec(sentences, vector_size=50, window=3, min_count=1, sg=0)  # Word2Vec 임베딩 모델입니다. vector_size는 벡터 크기입니다.

print("학교와 가장 유사한 단어:", model.wv.most_similar("학교", topn=1))  # 결과를 화면에 출력합니다.

```
</details>

In [ ]:
# 여기에 작성하세요
from gensim.models import Word2Vec  # Word2Vec 임베딩 모델입니다.

sentences = [
    ["나는", "학교에", "간다"],
    ["학교에서", "공부한다"],
    ["나는", "밥을", "먹었다"],
    ["좋은", "학교", "수업"],
    ["맛있는", "밥"],
    ["시원한", "콜라를","마셨다"],
]

#### 문제 6. Word2Vec 벡터 연산
Word2Vec을 학습한 모델에서 `"밥"` - `"먹었다"` + `"마셨다"` ≈ ? 를 계산해보세요.  

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

print(model.wv.most_similar(positive=["밥", "마셨다"], negative=["먹었다"], topn=1))  # 결과를 화면에 출력합니다.

```
</details>


In [ ]:
# 여기에 작성하세요


#### 문제 7. Word2Vec 학습 방식 바꾸기

같은 문장 리스트로 `sg=1`을 지정해 Skip-Gram 방식의 Word2Vec 모델을 학습해 보세요. 학습한 뒤 `"밥"`과 가장 유사한 단어를 확인하세요.

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

skipgram_model = Word2Vec(sentences, vector_size=50, window=3, min_count=1, sg=1)  # sg=1은 Skip-Gram 방식입니다.

print("밥과 가장 유사한 단어:", skipgram_model.wv.most_similar("밥", topn=1))  # 결과를 화면에 출력합니다.
```
</details>


In [ ]:
# 여기에 작성하세요


## 5. 텍스트 분류 미니 프로젝트

짧은 리뷰 문장을 사용해 CountVectorizer와 TF-IDF가 문장을 어떤 숫자 행렬로 바꾸는지 확인합니다.

- 진행 내용: `전처리 -> 벡터화 -> 간단한 해석` 흐름을 확인합니다.
- 사용 도구: `CountVectorizer`, `TfidfVectorizer`, `LogisticRegression`

<img src="image/text_classification_pipeline.svg" width="760">

이미지 출처: 김민수 강사


In [ ]:
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.

reviews = [
    ("배송이 빨라서 만족합니다", 1),
    ("포장이 깔끔하고 품질도 좋아요", 1),
    ("생각보다 성능이 좋아서 추천합니다", 1),
    ("가격 대비 괜찮은 선택이었어요", 1),
    ("다시 사고 싶지 않을 만큼 아쉬워요", 0),
    ("설명과 달라서 많이 실망했습니다", 0),
    ("배송이 늦고 응대도 아쉬웠어요", 0),
    ("품질이 기대보다 떨어집니다", 0),
]

df = pd.DataFrame(reviews, columns=["text", "label"])  # 데이터프레임을 직접 만듭니다.
df


### 문제 8. CountVectorizer로 문장 행렬 만들기

위의 `df["text"]`를 `CountVectorizer`로 변환하고, 단어 목록과 문서-단어 행렬을 확인해 보세요.

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.feature_extraction.text import CountVectorizer  # 문장을 단어 개수 벡터로 바꾸는 도구입니다.

count_vectorizer = CountVectorizer()  # 단어 빈도 벡터화 객체입니다.
X_count = count_vectorizer.fit_transform(df["text"])  # 학습과 변환을 한 번에 수행합니다.

print(count_vectorizer.get_feature_names_out())  # 결과를 화면에 출력합니다.
print(X_count.toarray())  # 결과를 화면에 출력합니다.

```
</details>


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer  # 단어 빈도 벡터화 도구입니다.

# 여기에 작성하세요


### 문제 9. TfidfVectorizer로 다시 표현하기

같은 데이터를 `TfidfVectorizer`로 바꾸고, `CountVectorizer`와 어떤 차이가 보이는지 확인해 보세요.

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.feature_extraction.text import TfidfVectorizer  # 문장을 TF-IDF 벡터로 바꾸는 도구입니다.

tfidf_vectorizer = TfidfVectorizer()  # TF-IDF 벡터화 객체입니다.
X_tfidf = tfidf_vectorizer.fit_transform(df["text"])  # 학습과 변환을 한 번에 수행합니다.

print(tfidf_vectorizer.get_feature_names_out())  # 결과를 화면에 출력합니다.
print(X_tfidf.toarray())  # 결과를 화면에 출력합니다.

```
</details>


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF 벡터화 도구입니다.

# 여기에 작성하세요


### 추가 문제. 분류 결과에 영향을 많이 준 단어 확인하기

TF-IDF로 벡터화한 뒤 로지스틱 회귀를 학습하고, 긍정/부정 예측에 큰 영향을 준 단어를 확인해 보세요.

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.feature_extraction.text import TfidfVectorizer  # 문장을 TF-IDF 벡터로 바꾸는 도구입니다.
from sklearn.linear_model import LogisticRegression  # 단어 영향도를 확인할 선형 분류 모델입니다.

vectorizer = TfidfVectorizer()  # 문장을 숫자 벡터로 바꾸는 객체입니다.
X = vectorizer.fit_transform(df["text"])  # 학습과 변환을 한 번에 수행합니다.
y = df["label"]  # 정답 값 또는 두 번째 배열을 준비합니다.

clf = LogisticRegression(max_iter=1000, random_state=42)  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
clf.fit(X, y)  # 데이터로 모델이나 변환 기준을 학습합니다.

feature_names = vectorizer.get_feature_names_out()  # 벡터화된 단어 목록을 가져옵니다.
coef = clf.coef_[0]  # 각 단어의 분류 가중치를 가져옵니다.

top_positive = coef.argsort()[-5:][::-1]  # 긍정 방향 가중치가 큰 단어 인덱스입니다.
top_negative = coef.argsort()[:5]  # 부정 방향 가중치가 큰 단어 인덱스입니다.

print("긍정에 가까운 단어:", feature_names[top_positive])  # 결과를 화면에 출력합니다.
print("부정에 가까운 단어:", feature_names[top_negative])  # 결과를 화면에 출력합니다.

```
</details>


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer  # 문장을 TF-IDF 벡터로 바꾸는 도구입니다.
from sklearn.linear_model import LogisticRegression  # 단어 영향도를 확인할 선형 분류 모델입니다.

# 여기에 작성하세요


## 6. 실제 영화리뷰 감성 분류 실습: NSMC

이번에는 실제 리뷰 데이터인 **네이버 영화리뷰 감성분석 데이터(NSMC)** 를 사용해 감성 분류 모델을 만들어 봅니다.

- 원본 데이터: https://github.com/e9t/nsmc  
- 데이터 파일: `data/ratings_train.txt`, `data/ratings_test.txt`  
- 입력 데이터: 영화 리뷰 문장  
- 정답 라벨: `0`은 부정, `1`은 긍정  
- 진행 내용: 간단한 EDA와 정제를 거친 뒤, TF-IDF 벡터와 로지스틱 회귀로 새 리뷰의 감성을 예측합니다.

실제 리뷰는 표현이 다양하므로 TF-IDF 벡터의 크기가 너무 커질 수 있습니다. 그래서 `max_features`, `min_df`, `max_df`로 벡터 크기를 통제합니다.


### 6.1 데이터 불러오기

먼저 학습용 파일과 테스트용 파일을 각각 읽어 옵니다. NSMC 파일은 열이 탭(`\t`)으로 구분된 텍스트 파일입니다.


In [ ]:
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.

# read_csv는 CSV뿐 아니라 구분자가 있는 텍스트 파일도 읽을 수 있습니다.
# sep="\t"는 열이 쉼표가 아니라 탭 문자로 구분되어 있다는 뜻입니다.
train_df = pd.read_csv("data/ratings_train.txt", sep="\t")  # 학습용 리뷰 데이터입니다.
test_df = pd.read_csv("data/ratings_test.txt", sep="\t")  # 최종 평가용 리뷰 데이터입니다.


In [ ]:
# head()는 데이터 앞부분 5행을 보여줍니다.
# id: 리뷰 번호, document: 리뷰 문장, label: 정답 라벨입니다.
train_df.head()


### 6.2 간단한 EDA

모델을 만들기 전에 데이터 크기, 결측치, 중복, 라벨 분포를 확인합니다. 텍스트 데이터에서는 빈 리뷰나 중복 리뷰가 섞이는 경우가 많습니다.


In [ ]:
# shape는 데이터프레임의 (행 개수, 열 개수)를 알려줍니다.
print("train 크기:", train_df.shape)
print("test 크기:", test_df.shape)


In [ ]:
# DataFrame({...})은 딕셔너리 형태의 데이터를 표로 만드는 문법입니다.
# len(...)은 행 개수, isna().sum()은 결측치 개수, duplicated().sum()은 중복 개수를 셉니다.
summary = pd.DataFrame({
    "train": [
        len(train_df),  # 전체 학습 데이터 행 수입니다.
        train_df["document"].isna().sum(),  # 리뷰 문장이 비어 있는 행 수입니다.
        train_df["document"].duplicated().sum(),  # 같은 리뷰 문장이 반복된 행 수입니다.
        (train_df["label"] == 1).sum(),  # label이 1인 긍정 리뷰 수입니다.
        (train_df["label"] == 0).sum(),  # label이 0인 부정 리뷰 수입니다.
    ],
    "test": [
        len(test_df),
        test_df["document"].isna().sum(),
        test_df["document"].duplicated().sum(),
        (test_df["label"] == 1).sum(),
        (test_df["label"] == 0).sum(),
    ],
}, index=["전체 행", "리뷰 결측", "리뷰 중복", "긍정 리뷰", "부정 리뷰"])

summary


In [ ]:
# map({0: "부정", 1: "긍정"})은 숫자 라벨을 사람이 읽기 쉬운 글자로 바꿉니다.
# value_counts()는 값별 개수를 세어 라벨 균형을 확인할 때 자주 씁니다.
label_counts = train_df["label"].map({0: "부정", 1: "긍정"}).value_counts()
label_counts


In [ ]:
# plot(kind="bar")는 막대그래프를 그립니다.
# rot=0은 x축 글자가 기울어지지 않도록 설정합니다.
label_counts.plot(kind="bar", rot=0)
plt.title("NSMC train label distribution")
plt.xlabel("label")
plt.ylabel("count")
plt.show()


### 6.3 텍스트 정제

리뷰에는 특수문자, 반복 기호, 빈 문자열이 섞여 있습니다. 여기서는 모델 실습에 필요한 최소 정제만 적용합니다.


In [ ]:
import re  # 정규표현식을 사용하기 위한 모듈입니다.

# def는 함수를 만드는 문법입니다. 같은 정제 과정을 여러 문장에 반복 적용하기 위해 함수로 묶습니다.
def clean_review(text):
    text = str(text)  # 혹시 숫자나 결측값이 들어와도 문자열로 바꿔 처리합니다.
    # re.sub(패턴, 바꿀값, 텍스트)는 패턴에 맞는 부분을 다른 값으로 바꿉니다.
    # 아래 패턴은 한글, 영문, 숫자, 공백이 아닌 문자를 공백으로 바꿉니다.
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", text)
    # \s+는 공백이 1개 이상 반복되는 경우를 뜻합니다. 여러 공백을 하나로 줄입니다.
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [ ]:
# 정제 함수가 어떻게 동작하는지 짧은 예시로 먼저 확인합니다.
sample_text = "와... 이 영화ㅋㅋ 진짜!!! 최고네요~~~"
clean_review(sample_text)


In [ ]:
# dropna(subset=[...])는 지정한 열에 결측치가 있는 행을 제거합니다.
# drop_duplicates(subset=[...])는 지정한 열 기준으로 중복 행을 제거합니다.
# copy()는 원본 데이터프레임을 건드리지 않고 새로운 복사본을 만들기 위해 사용합니다.
train_clean = train_df.dropna(subset=["document"]).drop_duplicates(subset=["document"]).copy()
test_clean = test_df.dropna(subset=["document"]).drop_duplicates(subset=["document"]).copy()

print("정제 전 train 행 수:", len(train_df))
print("결측/중복 제거 후 train 행 수:", len(train_clean))


In [ ]:
# map(함수)는 Series의 각 값에 함수를 하나씩 적용합니다.
# document 원문을 정제한 결과를 text라는 새 열에 저장합니다.
train_clean["text"] = train_clean["document"].map(clean_review)
test_clean["text"] = test_clean["document"].map(clean_review)

# str.len()은 문자열 길이를 계산합니다.
# 길이가 0인 리뷰는 모델에 사용할 정보가 없으므로 제거합니다.
train_clean = train_clean[train_clean["text"].str.len() > 0].reset_index(drop=True)
test_clean = test_clean[test_clean["text"].str.len() > 0].reset_index(drop=True)

# label_name은 그래프나 표에서 라벨을 읽기 쉽게 보기 위한 보조 열입니다.
train_clean["label_name"] = train_clean["label"].map({0: "부정", 1: "긍정"})
train_clean["review_length"] = train_clean["text"].str.len()

train_clean[["document", "text", "label_name"]].head()


In [ ]:
# groupby("label_name")은 라벨별로 데이터를 묶습니다.
# describe()는 개수, 평균, 표준편차, 사분위수 같은 요약 통계를 보여줍니다.
train_clean.groupby("label_name")["review_length"].describe().round(1)


### 6.4 TF-IDF 벡터 크기 통제하기

실제 리뷰 데이터는 단어와 표현이 매우 다양합니다. `ngram_range=(1, 2)`를 무제한으로 사용하면 벡터 차원이 과하게 커질 수 있으므로 아래 옵션으로 크기를 제한합니다.

| 옵션 | 의미 |
|------|------|
| `max_features=15000` | 최대 15,000개 표현만 사용합니다. |
| `min_df=3` | 3개 미만 문서에만 등장한 너무 드문 표현은 제거합니다. |
| `max_df=0.9` | 전체 문서의 90% 이상에 등장하는 너무 흔한 표현은 제거합니다. |
| `ngram_range=(1, 2)` | 단어 1개와 2개 조합까지만 사용합니다. |

수업 시간을 고려해 기본 학습 데이터는 50,000개로 제한합니다. 더 정확한 모델을 만들고 싶다면 `train_size`를 늘려 보세요.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer  # 문장을 TF-IDF 벡터로 바꾸는 도구입니다.
from sklearn.linear_model import LogisticRegression  # 분류에 사용할 로지스틱 회귀 모델입니다.
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report  # 평가 결과를 확인하는 도구입니다.


In [ ]:
# 전체 데이터를 모두 써도 되지만, 수업 실습에서는 실행 시간을 줄이기 위해 일부만 사용합니다.
# min(a, b)는 a와 b 중 더 작은 값을 선택합니다. 데이터가 부족해도 오류가 나지 않게 합니다.
train_size = min(50_000, len(train_clean))
test_size = min(20_000, len(test_clean))

# sample(n=...)은 데이터에서 n개 행을 무작위로 뽑습니다.
# random_state는 무작위 추출 결과가 매번 같아지도록 고정하는 값입니다.
train_model_df = train_clean.sample(n=train_size, random_state=42).reset_index(drop=True)
test_model_df = test_clean.sample(n=test_size, random_state=42).reset_index(drop=True)

print("학습에 사용할 리뷰 수:", len(train_model_df))
print("평가에 사용할 리뷰 수:", len(test_model_df))


In [ ]:
# 먼저 TF-IDF 벡터화 객체를 따로 만듭니다.
# 이 객체는 텍스트에서 단어/표현 목록을 학습하고, 각 리뷰를 숫자 벡터로 바꿉니다.
tfidf_vectorizer = TfidfVectorizer(
    max_features=15_000,  # 벡터 차원을 최대 15,000개 표현으로 제한합니다.
    min_df=3,  # 3개 미만 리뷰에만 등장한 너무 드문 표현은 제외합니다.
    max_df=0.9,  # 전체 리뷰의 90% 이상에 등장한 너무 흔한 표현은 제외합니다.
    ngram_range=(1, 2),  # 단어 1개와 2개 조합을 함께 사용합니다.
)

# 그 다음 분류 모델을 따로 만듭니다.
sentiment_model = LogisticRegression(
    max_iter=1000,  # 최적화 반복 횟수입니다. 부족하면 수렴 경고가 날 수 있습니다.
    solver="liblinear",  # 작은/중간 규모의 선형 분류에 안정적인 최적화 방법입니다.
    random_state=42,
)

tfidf_vectorizer, sentiment_model


In [ ]:
# fit_transform은 학습 데이터에서 단어 사전을 학습하고, 학습 리뷰를 숫자 벡터로 바꿉니다.
X_train_tfidf = tfidf_vectorizer.fit_transform(train_model_df["text"])

# transform은 이미 학습한 단어 사전을 기준으로 평가 리뷰를 숫자 벡터로 바꿉니다.
# 평가 데이터에는 fit을 다시 하지 않습니다. 그래야 학습/평가 기준이 같아집니다.
X_test_tfidf = tfidf_vectorizer.transform(test_model_df["text"])

# fit(입력 벡터, 정답)은 로지스틱 회귀 모델을 학습합니다.
sentiment_model.fit(X_train_tfidf, train_model_df["label"])

# predict(입력 벡터)는 학습된 모델로 새 리뷰의 라벨을 예측합니다.
pred = sentiment_model.predict(X_test_tfidf)

pred[:10]  # 예측 결과 앞 10개만 확인합니다.


In [ ]:
# get_feature_names_out()은 TF-IDF가 만든 단어/표현 목록을 가져옵니다.
feature_count = len(tfidf_vectorizer.get_feature_names_out())

print("TF-IDF 벡터 차원:", feature_count)
print("accuracy:", accuracy_score(test_model_df["label"], pred))

# classification_report는 precision, recall, f1-score를 한 번에 보여줍니다.
print(classification_report(test_model_df["label"], pred, target_names=["부정", "긍정"]))


In [ ]:
# ConfusionMatrixDisplay는 실제 라벨과 예측 라벨이 어떻게 맞고 틀렸는지 표 형태로 보여줍니다.
ConfusionMatrixDisplay.from_predictions(
    test_model_df["label"],
    pred,
    display_labels=["부정", "긍정"],
    cmap="Blues",
)
plt.title("NSMC TF-IDF + LogisticRegression")
plt.show()


### 6.5 직접 쓴 리뷰 예측하기

학습한 모델에 새로운 리뷰 문장을 넣어 긍정/부정을 예측해 봅니다.


In [ ]:
# 이 함수는 리뷰 한 문장을 받아 정제 -> 예측 -> 결과 정리까지 수행합니다.
def predict_review(review):
    cleaned = clean_review(review)  # 학습 데이터와 같은 방식으로 입력 문장을 정제합니다.
    # 새 리뷰도 학습 때 만든 TF-IDF 기준으로 숫자 벡터로 바꿔야 합니다.
    review_tfidf = tfidf_vectorizer.transform([cleaned])
    # predict_proba는 각 클래스일 확률을 반환합니다. [0]은 첫 번째 문장의 결과를 꺼내는 뜻입니다.
    proba = sentiment_model.predict_proba(review_tfidf)[0]
    # proba[1]은 긍정 클래스(라벨 1)의 확률입니다. 0.5 이상이면 긍정으로 판단합니다.
    pred_label = int(proba[1] >= 0.5)
    return {
        "review": review,
        "prediction": "긍정" if pred_label == 1 else "부정",
        "positive_probability": round(float(proba[1]), 3),
    }


In [ ]:
sample_reviews = [
    "배우 연기는 좋았지만 스토리가 너무 지루했다",
    "시간 가는 줄 모르고 봤다 정말 추천한다",
    "돈이 아까운 영화였다",
    "음악과 연출이 좋아서 다시 보고 싶다",
]

# 리스트 컴프리헨션은 리스트 안에서 반복문을 짧게 쓰는 문법입니다.
# 아래 코드는 sample_reviews의 각 리뷰에 predict_review를 적용해 결과 리스트를 만듭니다.
pd.DataFrame([predict_review(review) for review in sample_reviews])


### 6.6 모델이 중요하게 본 표현 확인하기

로지스틱 회귀는 각 단어/표현에 가중치를 부여합니다. 가중치가 크면 긍정 예측에, 작으면 부정 예측에 더 강하게 작용합니다.


In [ ]:
# 학습된 TF-IDF 변환기와 로지스틱 회귀 모델에서 필요한 값을 직접 꺼냅니다.
feature_names = tfidf_vectorizer.get_feature_names_out()  # 모델이 사용한 단어/표현 목록입니다.
coef = sentiment_model.coef_[0]  # 로지스틱 회귀가 학습한 각 표현의 가중치입니다.

print("표현 개수:", len(feature_names))
print("가중치 개수:", len(coef))


In [ ]:
# argsort()는 값을 작은 순서대로 정렬했을 때의 인덱스를 반환합니다.
# coef.argsort()[-15:][::-1]은 가중치가 큰 표현 15개를 큰 순서대로 가져오는 문법입니다.
positive_index = coef.argsort()[-15:][::-1]
negative_index = coef.argsort()[:15]

top_positive = pd.DataFrame({
    "word": feature_names[positive_index],
    "weight": coef[positive_index],
})

top_negative = pd.DataFrame({
    "word": feature_names[negative_index],
    "weight": coef[negative_index],
})

print("긍정 예측에 영향을 많이 준 표현")
display(top_positive)

print("부정 예측에 영향을 많이 준 표현")
display(top_negative)


## 체크포인트

- 텍스트 전처리 방법은 데이터 목적에 따라 달라집니다.
- BoW, TF-IDF, Word2Vec은 모두 텍스트를 수치화하는 방법이지만 표현 방식과 장단점이 다릅니다.
- 딥러닝 모델을 쓰지 않아도 `벡터화 + 선형 모델`만으로 텍스트 분류의 기본 흐름을 만들 수 있습니다.
- 실제 리뷰 데이터에서는 `max_features`, `min_df`, `max_df`로 TF-IDF 벡터 크기를 통제해야 합니다.
